<i18n value="7d67b345-b680-4f39-a384-31655b390a78"/>

# 데이터 재구성 랩

이 랩에서는 각 사용자가 **`events`**에서 특정 작업을 수행한 횟수를 집계하는 **`clickpaths`** 테이블을 생성한 다음, 이 정보를 이전 노트북에서 생성한 **`transactions`**의 평면화된 뷰와 조인합니다.

또한, 품목 이름에서 추출한 정보를 기반으로 **`sales`**에 기록된 품목에 플래그를 지정하는 새로운 고차 함수를 살펴봅니다.

## 학습 목표
이 랩을 마치면 다음을 수행할 수 있어야 합니다.
- 테이블을 피벗 및 조인하여 각 사용자에 대한 클릭 경로 생성
- 고차 함수를 적용하여 구매한 제품 유형 플래그 지정

<i18n value="db657989-4a07-41c5-acc8-18d6ceabbc85"/>

## 설치 실행

설치 스크립트는 이 노트북의 나머지 부분을 실행하는 데 필요한 데이터를 생성하고 값을 선언합니다.

In [0]:
%run ../Includes/Classroom-Setup-04.9L

Python interpreter will be restarted.
Python interpreter will be restarted.


Resetting the learning environment:
| No action taken

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02"

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)

Creating & using the schema "3dt005_nuk5_da_dewd" in the catalog "hive_metastore"...(6 seconds)

Cloning the events table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/events...(26 seconds / 485,824 records)
Cloning the sales table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/sales...(6 seconds / 10,539 records)
Cloning the users table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/users...(5 seconds / 252,346 records)
Cloning the transactions table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/transactions...(5 seconds / 10,539 records)

Pre

<i18n value="3b76127f-ef49-4024-a970-67ac52a1fa63"/>

## 클릭 경로를 생성하기 위한 데이터세트 재구성
이 작업은 **`events`** 및 **`transactions`** 테이블의 데이터를 결합하여 사용자가 사이트에서 수행한 모든 작업과 최종 주문 내역에 대한 레코드를 생성합니다.

**`clickpaths`** 테이블에는 **`transactions`** 테이블의 모든 필드와 각 **`event_name`**의 개수가 각 열에 포함되어야 합니다. 구매를 완료한 각 사용자는 최종 테이블에 하나의 행을 가져야 합니다. 먼저 **`events`** 테이블을 피벗하여 각 **`event_name`**의 개수를 구해 보겠습니다.

<i18n value="a4124fa1-e5cc-467f-8a7c-fb5978fefad1"/>



### 1. **`events`**를 피벗하여 각 사용자의 작업 수를 계산합니다.
**`event_name`** 열에 지정된 각 사용자가 특정 이벤트를 수행한 횟수를 집계하려고 합니다. 이를 위해 **`user_id`**로 그룹화하고 **`event_name`**을 피벗하여 각 이벤트 유형의 개수를 각 열에 표시합니다. 그러면 아래 스키마가 생성됩니다.

| field | type | 
| --- | --- | 
| user_id | STRING |
| cart | BIGINT |
| pillows | BIGINT |
| login | BIGINT |
| main | BIGINT |
| careers | BIGINT |
| guest | BIGINT |
| faq | BIGINT |
| down | BIGINT |
| warranty | BIGINT |
| finalize | BIGINT |
| register | BIGINT |
| shipping_info | BIGINT |
| checkout | BIGINT |
| mattresses | BIGINT |
| add_item | BIGINT |
| press | BIGINT |
| email_coupon | BIGINT |
| cc_info | BIGINT |
| foam | BIGINT |
| reviews | BIGINT |
| original | BIGINT |
| delivery | BIGINT |
| premium | BIGINT |


아래에 이벤트 이름이 나와 있습니다.

In [0]:
-- TODO
CREATE OR REPLACE VIEW events_pivot
AS SELECT * FROM (
  SELECT user_id AS user, event_name
  FROM events
) PIVOT (
  count(event_name) FOR event_name IN
  ("cart", "pillows", "login", "main", "careers", "guest", "faq", "down", "warranty", "finalize", 
  "register", "shipping_info", "checkout", "mattresses", "add_item", "press", "email_coupon", 
  "cc_info", "foam", "reviews", "original", "delivery", "premium")
)

<i18n value="8646882a-e334-4293-ba4d-550e86a2ed79"/>

**참고**: 랩 전체에서 Python을 사용하여 가끔씩 검사를 실행합니다. 아래 도우미 함수는 지침을 따르지 않은 경우 변경해야 할 사항에 대한 메시지와 함께 오류를 반환합니다. 출력이 없으면 이 단계를 완료한 것입니다.

In [0]:
%python
def check_table_results(table_name, column_names, num_rows):
    assert spark.table(table_name), f"Table named **`{table_name}`** does not exist"
    assert spark.table(table_name).columns == column_names, "Please name the columns in the order provided above"
    assert spark.table(table_name).count() == num_rows, f"The table should have {num_rows} records"

<i18n value="75fd31d6-3674-48c4-bc6b-c321a47ade9b"/>

아래 셀을 실행하여 뷰가 올바르게 생성되었는지 확인하세요.

In [0]:
%python
event_columns = ['user', 'cart', 'pillows', 'login', 'main', 'careers', 'guest', 'faq', 'down', 'warranty', 'finalize', 'register', 'shipping_info', 'checkout', 'mattresses', 'add_item', 'press', 'email_coupon', 'cc_info', 'foam', 'reviews', 'original', 'delivery', 'premium']
check_table_results("events_pivot", event_columns, 204586)

<i18n value="bb41e6ea-aeae-4f4f-97c5-cad043d151cd"/>



### 2. 모든 사용자의 이벤트 수와 거래를 조인합니다.

다음으로, **`events_pivot`**과 **`transactions`**를 조인하여 **`clickpaths`** 테이블을 만듭니다. 이 테이블에는 위에서 만든 **`events_pivot`** 테이블과 동일한 이벤트 이름 열이 있어야 하며, 그 뒤에 **`transactions`** 테이블의 열이 있어야 합니다(아래 참조).

| field | type | 
| --- | --- | 
| user | STRING |
| cart | BIGINT |
| ... | ... |
| user_id | STRING |
| order_id | BIGINT |
| transaction_timestamp | BIGINT |
| total_item_quantity | BIGINT |
| purchase_revenue_in_usd | DOUBLE |
| unique_items | BIGINT |
| P_FOAM_K | BIGINT |
| M_STAN_Q | BIGINT |
| P_FOAM_S | BIGINT |
| M_PREM_Q | BIGINT |
| M_STAN_F | BIGINT |
| M_STAN_T | BIGINT |
| M_PREM_K | BIGINT |
| M_PREM_F | BIGINT |
| M_STAN_K | BIGINT |
| M_PREM_T | BIGINT |
| P_DOWN_S | BIGINT |
| P_DOWN_K | BIGINT |

In [0]:
-- TODO
CREATE OR REPLACE VIEW clickpaths AS
SELECT *
FROM events_pivot a
JOIN transactions b
ON a.user = b.user_id

<i18n value="db501e67-0444-4988-a850-e7f374b38f3e"/>

아래 셀을 실행하여 테이블이 올바르게 생성되었는지 확인하세요.

In [0]:
%python
clickpath_columns = event_columns + ['user_id', 'order_id', 'transaction_timestamp', 'total_item_quantity', 'purchase_revenue_in_usd', 'unique_items', 'P_FOAM_K', 'M_STAN_Q', 'P_FOAM_S', 'M_PREM_Q', 'M_STAN_F', 'M_STAN_T', 'M_PREM_K', 'M_PREM_F', 'M_STAN_K', 'M_PREM_T', 'P_DOWN_S', 'P_DOWN_K']
check_table_results("clickpaths", clickpath_columns, 9085)

<i18n value="ed22d836-9233-469c-8aa1-4bea42f84517"/>


## 구매한 제품 유형 플래그 지정
여기서는 고차 함수 **`EXISTS`**를 **`sales`** 테이블의 데이터와 함께 사용하여 구매한 품목이 매트리스인지 베개인지를 나타내는 부울 열 **`mattress`**와 **`pillow`**를 생성합니다.

예를 들어, **`items`** 열의 **`item_name`**이 문자열 **`"Mattress"`**로 끝나는 경우, **`mattress`**의 열 값은 **`true`**이고 **`pillow`**의 값은 **`false`**여야 합니다. 다음은 품목과 결과 값의 몇 가지 예입니다.

|  items  | mattress | pillow |
| ------- | -------- | ------ |
| **`[{..., "item_id": "M_PREM_K", "item_name": "Premium King Mattress", ...}]`** | true | false |
| **`[{..., "item_id": "P_FOAM_S", "item_name": "Standard Foam Pillow", ...}]`** | false | true |
| **`[{..., "item_id": "M_STAN_F", "item_name": "Standard Full Mattress", ...}]`** | true | false |

<a href="https://docs.databricks.com/sql/language-manual/functions/exists.html" target="_blank">exists</a> 함수에 대한 설명서를 참조하세요.
조건식 **`item_name LIKE "%Mattress"`**를 사용하면 문자열 **`item_name`**이 "Mattress"라는 단어로 끝나는지 확인할 수 있습니다.

In [0]:
-- TODO
CREATE OR REPLACE TABLE sales_product_flags AS
SELECT
  items,
  EXISTS(items, i -> i.item_name LIKE "%Mattress") AS mattress,
  EXISTS(items, i -> i.item_name LIKE "%Pillow") AS pillow
FROM sales

num_affected_rows,num_inserted_rows


<i18n value="3d56b9b4-957d-4ea2-8c49-f4e92a4918c9"/>

아래 셀을 실행하여 테이블이 올바르게 생성되었는지 확인하세요.

In [0]:
%python
check_table_results("sales_product_flags", ['items', 'mattress', 'pillow'], 10539)
product_counts = spark.sql("SELECT sum(CAST(mattress AS INT)) num_mattress, sum(CAST(pillow AS INT)) num_pillow FROM sales_product_flags").first().asDict()
assert product_counts == {'num_mattress': 10015, 'num_pillow': 1386}, "There should be 10015 rows where mattress is true, and 1386 where pillow is true"

<i18n value="e78c6d7f-bd25-4af8-8d01-a6943f24b8d6"/>

이 레슨과 관련된 표와 파일을 삭제하려면 다음 셀을 실행하세요.

In [0]:
%python
DA.cleanup()